# Kepler's Third Law

## The law

A planet farther from the Sun takes longer to go all the way around.

**Kepler's third law** makes that precise. If

- **P** is the time for one full orbit (the **period**),
- **a** is the **semi-major axis** — half the long width of the ellipse; for a circle, **a** is just the radius,

then

$$P^2 \propto a^3.$$

The square of the period grows with the cube of the orbit's size. Doubling the size does **not** double the period. It multiplies the period by $\sqrt{8} \approx 2.8$.

## Where the formula comes from

For a planet orbiting a fixed Sun of mass **M**, Newton’s gravity gives

$$P = 2\pi \sqrt{\frac{a^3}{GM}}.$$

**G** is the gravitational constant. Squaring both sides,

$$P^2 = \frac{4\pi^2}{GM}\, a^3.$$

Around the Sun, if **a** is measured in astronomical units (AU) and **P** in years, this becomes the simple line

$$P^2 = a^3.$$

Earth sits near **a = 1 AU** and **P = 1 year**, so $1^2 = 1^3$.

## What this notebook does

Simulate several orbits of different sizes. Measure **a** and **P** from each simulation. Plot **P²** against **a³**. If the law holds, the points fall on the line **P² = a³**.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

from src.physics import AU, M_SUN, YEAR, circular_orbital_speed, kepler_period
from src.simulation import (
    KeplerLawSample,
    integrate_two_body,
    orbit_elements_from_history,
)
from src.visualization import apply_dark_style, plot_kepler_third_law

%matplotlib inline
apply_dark_style()


## 1. Several orbits, different sizes

Each run starts the planet on the +x axis at a chosen distance. The sideways speed is **90% of circular speed** at that distance, so the path is a mild ellipse rather than a perfect circle. A circle has no clear closest point, and the period is easier to measure from one closest approach to the next.

The Sun stays fixed at the origin. Only its gravity acts on the planet.


In [ ]:
start_radii_au = [0.6, 0.8, 1.0, 1.4, 1.8]
dt = 6 * 3600.0  # 6 hours, in seconds
samples = []

for r_au in start_radii_au:
    r0 = r_au * AU
    v_circ = circular_orbital_speed(r0, M_SUN)
    pos0 = np.array([r0, 0.0, 0.0])
    vel0 = np.array([0.0, 0.9 * v_circ, 0.0])

    # Long enough to see at least two closest approaches.
    n_steps = int(2.8 * kepler_period(r0, M_SUN) / dt)
    history = integrate_two_body(pos0, vel0, M_SUN, dt, n_steps)
    elements = orbit_elements_from_history(history)
    samples.append(
        KeplerLawSample(
            semi_major_axis=elements["semi_major_axis_energy"],
            period=elements["period_simulated"],
            central_mass=M_SUN,
        )
    )

print(f"{'a (AU)':>10} {'P sim (yr)':>12} {'P formula (yr)':>16}")
for sample in samples:
    a_au = sample.semi_major_axis / AU
    p_sim = sample.period / YEAR
    p_formula = kepler_period(sample.semi_major_axis, M_SUN) / YEAR
    print(f"{a_au:10.3f} {p_sim:12.3f} {p_formula:16.3f}")


## 2. Does P² equal a³?

Each point is one simulated orbit. The dashed line is **P² = a³**. The solid line is a straight fit through the points. For the Sun, in these units, the slope should be about **1**.


In [ ]:
import matplotlib.pyplot as plt

fig = plot_kepler_third_law(samples)
plt.show()


## 3. What this does and does not show

The points follow the line, so the simulation agrees with Kepler's third law for these orbits.

This check assumes:

- The Sun is fixed and much heavier than the planet.
- Gravity is Newton's inverse-square law. There is no drag and no third body.
- **a** comes from the orbit's energy, and **P** comes from the time between closest approaches.

A different star mass would change the slope. The simple form **P² = a³** is special to measuring orbits around the Sun in years and AU.
